# NumPy Practice — Week 1
**Aayan Mulani | Decimal Point Analytics Preparation**

NumPy (Numerical Python) is the core library for numerical computation in Python. It provides the `ndarray` — a fast, memory-efficient array object — and a large collection of mathematical functions that operate on entire arrays at once.

**Why NumPy for finance?**
Financial analysis involves large datasets — thousands of daily prices, returns across hundreds of stocks. Python's built-in lists are slow for this. NumPy arrays are up to 50x faster because they are stored in contiguous memory and operations run in optimised C code under the hood.

## 1. Importing NumPy
`import numpy as np` is a universal convention — every data scientist writes it this way. The alias `np` saves typing and makes code more readable.

In [ ]:
import numpy as np

## 2. Creating Arrays
A NumPy **array** (ndarray) is like a Python list but faster and more powerful for math. All elements in a NumPy array must be the same data type.

| Function | Description | Finance Use |
|---|---|---|
| `np.array()` | Convert list to array | Store price series |
| `np.zeros()` | Array of zeros | Initialise empty return array |
| `np.ones()` | Array of ones | Equal weight portfolio |
| `np.arange()` | Evenly spaced integers | Generate year sequence |
| `np.linspace()` | Evenly spaced floats | Generate probability levels |

In [ ]:
print(np.array([1, 2, 3, 4, 5]))    # from a list
print(np.zeros(5))                  # five zeros
print(np.ones((3, 3)))             # 3x3 matrix of ones
print(np.arange(0, 10, 2))        # 0 to 8, step 2
print(np.linspace(0, 1, 5))       # 5 points evenly spaced between 0 and 1

## 3. Array Arithmetic — Vectorisation
**Vectorisation** means applying an operation to every element of an array simultaneously, without writing a loop. This is one of NumPy's most important features.

In finance: if you have 500 stock prices and want to compute daily returns, vectorisation does it in one line instead of a 500-iteration loop.

In [ ]:
a = np.array([10, 20, 30, 40, 50])  # imagine these are stock prices

print(a * 2)     # double every price
print(a + 100)   # add 100 to every price
print(a ** 2)    # square every price
print(a / 10)    # divide every price by 10

## 4. Indexing and Slicing
Accessing elements works the same as Python lists — zero-indexed, supports slicing and negative indexing.

**Boolean indexing** is unique to NumPy and extremely powerful. `a[a > 20]` returns only elements where the condition is True. In finance, this is how you filter: *'give me all days where the return exceeded 2%'* — in one line, no loop needed.

In [ ]:
a = np.array([10, 20, 30, 40, 50])

print(a[0])       # first element
print(a[-1])      # last element
print(a[1:4])     # elements at index 1, 2, 3
print(a[a > 20])  # boolean indexing — only values greater than 20

## 5. Statistical Functions
NumPy provides fast implementations of all key statistics. These are the same metrics used in portfolio analysis and risk management.

| Function | Description | Finance Meaning |
|---|---|---|
| `np.mean()` | Average | Average daily return |
| `np.std()` | Standard deviation | Volatility (daily) |
| `np.max()` | Maximum value | 52-week high |
| `np.min()` | Minimum value | 52-week low |
| `np.sum()` | Sum of all values | Total return |
| `np.cumsum()` | Running total | Cumulative portfolio value |

In [ ]:
a = np.array([10, 20, 30, 40, 50])

print(np.mean(a))    # 30.0
print(np.std(a))     # 14.14 — spread around the mean
print(np.max(a))     # 50
print(np.min(a))     # 10
print(np.sum(a))     # 150
print(np.cumsum(a))  # [10, 30, 60, 100, 150] — running total

## 6. Financial Application — Portfolio Statistics
This function takes an array of daily returns and computes four key portfolio metrics.

**Why multiply by 252?** There are approximately 252 trading days in a year (markets are closed on weekends and public holidays). To convert daily metrics to annual:
- Annualised Return = Daily Mean × 252
- Annualised Volatility = Daily Std Dev × √252

**Why √252 and not 252 for volatility?** Volatility scales with the square root of time — a statistical property. Doubling the time period does not double the risk; it increases it by a factor of √2.

In [ ]:
def portfolio_stats(returns):
    """
    Compute key portfolio statistics from daily returns.
    returns : NumPy array of daily returns (as decimals, e.g. 0.01 = 1%)
    """
    mean_return   = np.mean(returns)
    std_return    = np.std(returns)
    ann_return    = mean_return * 252
    ann_volatility = std_return * np.sqrt(252)

    print(f'Mean Daily Return:      {mean_return:.4f}  ({mean_return*100:.2f}%)')
    print(f'Daily Std Dev:          {std_return:.4f}  ({std_return*100:.2f}%)')
    print(f'Annualised Return:      {ann_return:.4f}  ({ann_return*100:.2f}%)')
    print(f'Annualised Volatility:  {ann_volatility:.4f}  ({ann_volatility*100:.2f}%)')

# Sample daily returns — mix of gains and losses
sample_returns = np.array([0.01, -0.005, 0.02, -0.01, 0.015, 0.008, -0.003])
portfolio_stats(sample_returns)

## 7. NumPy vs Loop — Speed Comparison
The moving average is a common financial indicator. Here we compare two implementations:
1. **Loop version** — readable but slow (pure Python)
2. **NumPy version** — uses `np.convolve()`, runs in C code

`np.convolve(prices, weights, mode='valid')` slides the weights window across the prices array and computes a weighted sum at each position. For equal weights (all = 1/window), this is the simple moving average.

`%timeit` is a Jupyter magic command that runs the code many times and reports average execution time.

In [ ]:
def moving_average_loop(prices, window):
    """Moving average using a Python loop — simple but slow."""
    result = []
    for i in range(window - 1, len(prices)):
        avg = sum(prices[i - window + 1 : i + 1]) / window
        result.append(avg)
    return result

def moving_average_numpy(prices, window):
    """Moving average using np.convolve — fast vectorised implementation."""
    weights = np.ones(window) / window
    return np.convolve(np.array(prices), weights, mode='valid')

big_prices = list(range(10000))

%timeit moving_average_loop(big_prices, 50)
%timeit moving_average_numpy(big_prices, 50)

**Key takeaway:** NumPy is ~10x faster on 10,000 data points. In production environments with millions of rows, this difference becomes critical for performance.